In [1]:
from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers")

In [14]:
import pandas as pd
df = dataset["train"].to_pandas()
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='str')

In [15]:
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [17]:
df = df[['title', 'abstract']]
df=df.head(50000)
df.isnull().sum()

title       0
abstract    0
dtype: int64

To merge the title and abstract into a single, context-rich text block so sentence transformer can generate one comprehensive embedding per research paper.

In [18]:
df["paper_text"] = df["title"] + " " + df["abstract"]
df

,title,abstract,paper_text
0,Learning from compressed observations,The problem of statistical learning is to co...,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun...",Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...,Parametric Learning and Monte Carlo Optimizati...
...,...,...,...
49995,"See, Attend and Brake: An Attention-based Sali...",Visual perception is the most critical input...,"See, Attend and Brake: An Attention-based Sali..."
49996,SNIFF: Reverse Engineering of Neural Networks ...,Neural networks have been shown to be vulner...,SNIFF: Reverse Engineering of Neural Networks ...
49997,Beyond Dropout: Feature Map Distortion to Regu...,Deep neural networks often consist of a grea...,Beyond Dropout: Feature Map Distortion to Regu...
49998,A study of resting-state EEG biomarkers for de...,Background: Depression has become a major he...,A study of resting-state EEG biomarkers for de...


In [19]:
df['paper_text'].head()

0    Learning from compressed observations   The pr...
1    Sensor Networks with Random Links: Topology De...
2    The on-line shortest path problem under partia...
3    A neural network approach to ordinal regressio...
4    Parametric Learning and Monte Carlo Optimizati...
Name: paper_text, dtype: str

In [20]:
df[["paper_text"]].head()


,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [21]:
print(df['paper_text'].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

### **The Objective**

To load the specific neural network model that will actually perform the core task of your project: converting your cleaned text into mathematical vector embeddings.

In [22]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [24]:
sample_text = df["paper_text"].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate functions. The ideas are\nillustrated on the example of nonparam

### **The Breakdown & Explanation**

* **`df["paper_text"]`**: You are targeting that "golden" combined text column you created a few steps ago.
* **`.iloc[0]`**: This stands for "integer location." In pandas, this is the standard, most efficient way to grab data by its exact mathematical row number. `[0]` targets the very first row.
* **Summary:** "Before processing a massive batch of data through a neural network, it is an industry best practice to isolate a single sample (like index 0) and pass it through the pipeline to verify the logic and output shape. It saves hours of potential debugging."

### **The Output**

Once you run this, the variable `sample_text` will hold a plain Python string containing the title and abstract of the very first research paper in your DataFrame. It is now perfectly prepped to be fed directly into your loaded `model` in the next cell.

In [25]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()

In [26]:
sample_text = df["paper_text"].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regres

In [27]:
embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [28]:
embedding[:56]

array([-0.13156407, -0.00678268, -0.00367611,  0.03265155,  0.11219642,
        0.01227268,  0.09816721, -0.09005226,  0.0423116 , -0.01977348,
       -0.03308415,  0.07452946,  0.10632039, -0.02060432, -0.020521  ,
        0.00169492,  0.07081953,  0.05854454, -0.11231905,  0.02082477,
        0.05692547,  0.02015781,  0.02583109,  0.03217028,  0.10513762,
       -0.09676762,  0.02700801, -0.02345093, -0.04549678, -0.010137  ,
       -0.01794856, -0.04814427,  0.01077651, -0.03759068,  0.01943482,
        0.03715186,  0.02967846,  0.04330938,  0.04373212,  0.03704865,
       -0.00182595,  0.00455185, -0.00799064,  0.03037369, -0.01437797,
        0.03795145,  0.05959158, -0.02583358, -0.06521577,  0.05900269,
       -0.02107134,  0.07359419, -0.05720105,  0.0029406 ,  0.00767514,
       -0.03334163], dtype=float32)

In [29]:
sample_embedding = model.encode(
    df["paper_text"].head(5).to_list()
)

print(sample_embedding.shape)

(5, 384)


In [30]:
import sklearn
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
similarity = cosine_similarity([sample_embedding[0]], [sample_embedding[0]])
print(similarity)

[[1.0000001]]


In [32]:
similarity = cosine_similarity(sample_embedding[0].reshape(1, -1), sample_embedding[1].reshape(1, -1))
print(similarity)

[[0.36625272]]


In [33]:
# 1. Run the import again to fix the NameError
from sklearn.metrics.pairwise import cosine_similarity

# 2. Run your loop
for i in range(1, 5):
    sim = cosine_similarity(sample_embedding[0].reshape(1, -1), sample_embedding[i].reshape(1, -1))
    print(f"Similarity with Paper {i}: {sim[0][0]}")

Similarity with Paper 1: 0.36625272035598755
Similarity with Paper 2: 0.33522844314575195
Similarity with Paper 3: 0.1550510972738266
Similarity with Paper 4: 0.37421542406082153


In [34]:
embedding = model.encode(df["paper_text"].to_list(), batch_size=32, show_progress_bar=True)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [ ]:
import numpy as np
import os
index_path = "../data/arxiv_embeddings.npy"

if os.path.exists(index_path):
    print("Loading saved embeddings")
    embeddings = np.load(index_path)
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save(index_path, embeddings)
    print("Embeddings saved successfully!")
# This saves your 26-minute calculation instantly to a file in your folder
np.save(index_path, embedding)

print("Embeddings successfully saved to disk!")

Embeddings successfully saved to disk!


In [ ]:
# Save the cleaned dataframe so we can map the math back to actual paper titles
df.to_csv("../data/cleaned_arxiv_papers.csv", index=False)
print("Cleaned DataFrame saved successfully!")

Cleaned DataFrame saved successfully!
